In [ ]:
import numpy as np
import pandas as pd

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
import pandas as pd


file_path = '/content/drive/MyDrive/ratings_long.csv'


df = pd.read_csv(file_path)
print(df.head())

   userId  movieId  rating
0       0       16       5
1       0       72       5
2       0       86       5
3       0      259       1
4       0      319       4


In [24]:
r = np.full((20, 1000),fill_value=np.nan)

In [25]:
for rec in df.itertuples():
    r[rec.userId][rec.movieId] = rec.rating

In [26]:
r

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan,  4., nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]])

In [27]:
import numpy as np
import pandas as pd
import torch

r = np.full((20, 1000), fill_value=np.nan)

for rec in df.itertuples():
    r[int(rec.userId), int(rec.movieId)] = rec.rating

R = torch.tensor(r, dtype=torch.float32)
mask = ~torch.isnan(R)

u = torch.randn(20, 4, requires_grad=True)
v = torch.randn(4, 1000, requires_grad=True)

learning_rate = 0.01
lmbda = 0.01
epochs = 2000

for epoch in range(epochs):
    prediction = torch.mm(u, v)

    error = torch.sum((R[mask] - prediction[mask]) ** 2)
    reg = lmbda * (torch.sum(u**2) + torch.sum(v**2))
    loss = error + reg

    loss.backward()

    with torch.no_grad():
        u -= learning_rate * u.grad
        v -= learning_rate * v.grad

        u.grad.zero_()
        v.grad.zero_()

    if epoch % 200 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 3174.1914
Epoch 200, Loss: 40.5519
Epoch 400, Loss: 37.7272
Epoch 600, Loss: 35.1221
Epoch 800, Loss: 32.7195
Epoch 1000, Loss: 30.5035
Epoch 1200, Loss: 28.4597
Epoch 1400, Loss: 26.5746
Epoch 1600, Loss: 24.8359
Epoch 1800, Loss: 23.2321


In [28]:
def tahmin_et(user_idx, movie_idx):

    with torch.no_grad():
        tahmin = torch.matmul(u[user_idx], v[:, movie_idx])
    return tahmin.item()

print(f"Tahmin edilen puan: {tahmin_et(0, 16):.2f}")

Tahmin edilen puan: 5.00


In [29]:

with torch.no_grad():
    tum_tahminler = torch.mm(u, v)


gercek_puanlar = R[mask]
tahmin_edilen_puanlar = tum_tahminler[mask]


mse = torch.mean((gercek_puanlar - tahmin_edilen_puanlar) ** 2)
rmse = torch.sqrt(mse)

print(f"Modelin toplam hata payı (RMSE): {rmse.item():.4f}")

Modelin toplam hata payı (RMSE): 0.0032
